# Manual Image Cropping (ROIs)

Companion notebook to `manual_cropping.py`: opens the cropping window, applies ROIs in
batch and checks the crops that were produced.

## Workflows

| Situation | What to do |
|---|---|
| Each photo has the sample in a different place | **Free selection**: drag the rectangle on each image and press `Enter` |
| You want every crop at the same size | Tick **Tamanho fixo**, enter `width x height` and click the centre of the region |
| The photos were taken from the same position (tripod/scanner) | Tick **Travar ROIs**, draw once and press `Enter` on each image, or use **Aplicar estas ROIs a TODAS as imagens** |
| You want several sub-images per region | Set the **grid** `rows x columns` inside the ROI |

Crops always come out at the **original resolution** (the zoom is only visual) and the
coordinates are stored in `crop_coords.csv`, which makes it possible to redo everything later.

**Shortcuts:** `Enter` saves and moves on | arrow keys change image | `Del` removes the last
ROI | `C` clears | `T` locks/unlocks | `+`/`-`/`0` zoom (or `Ctrl`+mouse wheel) | `F1` about.

> The window labels are in Portuguese, the language of the research group that uses the tool.

In [ ]:
from pathlib import Path

import manual_cropping as mc

# ==== adjust here ====
INPUT_FOLDER = Path('sample_image')     # folder with the original photographs
OUTPUT_FOLDER = Path('crops')           # where the crops will be written
IMAGE_FORMAT = 'png'                    # png keeps colour lossless (better for histograms)

images = mc.list_images(INPUT_FOLDER)
print(f'{len(images)} image(s) in {INPUT_FOLDER.resolve()}')
for path in images[:10]:
    print(' -', path.name)

## 1. Open the cropping window

The cell below opens the Tkinter window. **The notebook stays blocked while the window is
open** - close it for the kernel to respond again.

To work outside Jupyter (more stable), run in the terminal:

```powershell
python manual_cropping.py --input sample_image --output crops
```

In [ ]:
mc.open_window(INPUT_FOLDER, OUTPUT_FOLDER, IMAGE_FORMAT)

## 2. Fixed region applied to the whole folder (no window)

When the ROI is already known (for example, copied from the ROI list in the window or from
the CSV), everything can be cropped at once. Each ROI is `(x, y, width, height)` in pixels of
the original image.

In [ ]:
ROIS = [
    (1680, 1222, 285, 261),   # (x, y, width, height)
    # (300, 200, 600, 400),
]
N_ROWS, N_COLS = 2, 3   # each ROI becomes 2x3 = 6 sub-crops

total = mc.apply_in_batch(INPUT_FOLDER, OUTPUT_FOLDER, ROIS,
                          n_rows=N_ROWS, n_cols=N_COLS,
                          fmt=IMAGE_FORMAT, subfolder_per_image=False)

## 3. Check the crops that were produced

Reads `crop_coords.csv`, summarises how many crops came out of each image and shows a sample
mosaic. Different sizes between crops get in the way of comparing histograms, so it is worth
checking the dimension columns.

In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT_FOLDER / mc.CSV_NAME)
print(f'{len(df)} crops recorded')
display(df.groupby('image').agg(n_crops=('output_file', 'count'),
                                width=('width', 'unique'),
                                height=('height', 'unique')))
df.tail()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

sample = df['output_file'].tolist()[:12]
n_col = 4
n_lin = max(1, -(-len(sample) // n_col))
fig, axes = plt.subplots(n_lin, n_col, figsize=(3 * n_col, 3 * n_lin))
for axis, name in zip(fig.axes, sample):
    axis.imshow(Image.open(OUTPUT_FOLDER / name))
    axis.set_title(name, fontsize=7)
for axis in fig.axes:
    axis.axis('off')
plt.tight_layout()
plt.show()

## 4. Redo the crops from the CSV

Reproducibility: with the CSV stored alongside the paper, anyone reproduces exactly the same
crops (or regenerates them in another format/folder) without repeating the manual selection.
CSV files written by version 1.0, whose columns were in Portuguese, are also accepted.

In [ ]:
mc.redo_from_csv(OUTPUT_FOLDER / mc.CSV_NAME, INPUT_FOLDER,
                 output_folder=Path('crops_redone'), fmt='png')